In [1]:
from src.core.session import SparkFactory

spark = SparkFactory.create()

26/06/22 23:36:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
#Notebook — célula 1

from src.bronze.readers.file.csv_reader import CSVReader

In [3]:
#Célula 2 Escolha um CSV real.

source_path = "/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv"

In [4]:
#Célula 3 Teste

reader = CSVReader()

df = reader.read(source_path)

print(df.shape)

df.head()

(82, 58)


,projeto_area_de_operacao_vale,contrato_vigente,pacote,contrato,cnpj_fornecedor,conta_do_fornecedor_sap,fornecedor,objeto_contratual,prazo_analise,prazo_pagamento,...,data_envio_emissao,senha_de_recebimento,data_cadastro,data_vencimento,nota_paga,aguardando_pagamento,aguardando_bloqueio_abatimento,data_pagamento,observacao,arquivo
0,Implantação de Projetos Vitória,Sim,NaN,5900055119,32447237000101,90017911,FACOM F DE ALMEIDA CONSTRUCOES LTDA,ETEs Compactas - Complexo Tubarão,5 Dias,60 Dias,...,NaN,NaN,NaN,NaN,Não,Não,Não,NaN,NaN,NaN
1,Implantação de Projetos Vitória,Sim,NaN,5900079591,14992876000176,90036655,AUDIO VISUAL EDICOES LTDA,Filmagem e Time Lapse c Drone-VitóriaES,Dias,20 Dias,...,NaN,NaN,NaN,NaN,Não,Não,Não,NaN,NaN,NaN
2,Implantação de Projetos Vitória,Sim,NaN,5900080209,42773523000110,70002035,DALBEN CONSULTORIA EM ENGA ELETRICA TREINAMENT...,Cto 5500062696 - Dalben (Elétrica Vix),Dias,20 Dias,...,NaN,NaN,NaN,NaN,Não,Não,Não,NaN,NaN,NaN
3,Implantação de Projetos Vitória,Sim,NaN,5900086165,09181665000113,90056275,SERENG CONSULTING LTDA,FIS-SD-Gerenciamento Projetos Tubarão,Dias,60 Dias,...,NaN,NaN,NaN,NaN,Não,Não,Não,NaN,NaN,NaN
4,Implantação de Projetos Vitória,Sim,NaN,5900086957,05537906000163,70003374,SERENG - ENGENHARIA E CONSULTORIA,ENG - SD - Eng. Multidisciplinar VIX,Dias,60 Dias,...,NaN,NaN,NaN,NaN,Não,Não,Não,NaN,NaN,NaN


In [5]:
import importlib
import src.core.data_cleaner

importlib.reload(src.core.data_cleaner)

<module 'src.core.data_cleaner' from '/app/src/core/data_cleaner.py'>

In [6]:
from src.bronze.readers.file.csv_reader import CSVReader

reader = CSVReader()
df = reader.read(source_path)

df.columns[:5]

Index(['projeto_area_de_operacao_vale', 'contrato_vigente', 'pacote',
       'contrato', 'cnpj_fornecedor'],
      dtype='object')

In [7]:
import pandas as pd

raw_df = pd.read_csv(
    source_path,
    encoding="latin1",
    dtype=str,
    sep=None,
    engine="python"
)

print(raw_df.columns[0])

ï»¿PROJETO/ÃREA DE OPERAÃÃO VALE


In [8]:
print(repr(raw_df.columns[0]))

'ï»¿PROJETO/Ã\x81REA DE OPERAÃ\x87Ã\x83O VALE'


In [9]:
import pandas as pd

print(pd.__version__)

2.0.3


In [10]:
from src.core.session import SparkFactory
from src.bronze.readers.file.csv_reader import CSVReader
from src.bronze.converters.pandas_to_spark import PandasToSparkConverter

spark = SparkFactory.create()

source_path = "/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv"

reader = CSVReader()
df = reader.read(source_path)

spark_df = PandasToSparkConverter.convert(
    spark=spark,
    pandas_df=df
)

spark_df.printSchema()
spark_df.show(5)

root
 |-- projeto_area_de_operacao_vale: string (nullable = true)
 |-- contrato_vigente: string (nullable = true)
 |-- pacote: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cnpj_fornecedor: string (nullable = true)
 |-- conta_do_fornecedor_sap: string (nullable = true)
 |-- fornecedor: string (nullable = true)
 |-- objeto_contratual: string (nullable = true)
 |-- prazo_analise: string (nullable = true)
 |-- prazo_pagamento: string (nullable = true)
 |-- status_medicao: string (nullable = true)
 |-- n_do_bm: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- tecnico_de_medicao: string (nullable = true)
 |-- data_bm_preenchido: string (nullable = true)
 |-- data_analise_finalizada: string (nullable = true)
 |-- tempo_gasto_para_analise: string (nullable = true)
 |-- fiscal_nomeado: string (nullable = true)
 |-- data_aprovacao_fiscal: string (nullable = true)
 |-- tempo_gasto_p_aprovacao_do_fiscal: string (nullable = true)
 |-- administrador_vale:

26/06/22 23:36:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------+----------------+------+----------+---------------+-----------------------+--------------------+--------------------+-------------+---------------+--------------------+-------+--------------------+--------------------+-------------------+-----------------------+------------------------+--------------------+---------------------+---------------------------------+--------------------+----------------------------+-------------------------------------+--------------------+---------------------+------------------------------+---------------------------------+----------------------+-----+-----------------+-------------+----------------+----------------------+-----------+-----+---------------------+-------------------------------------------+----------------------+---------+---------------+----------------+---------------+-----------+-----------------------------+-----------+-----+---------+------+------------------+--------------------+-------------+-----------

In [11]:
#validar conversão Pandas → Spark.

from src.bronze.converters.pandas_to_spark import PandasToSparkConverter

spark_df = PandasToSparkConverter.convert(
    spark=spark,
    pandas_df=df
)

spark_df.printSchema()
spark_df.show(5)

root
 |-- projeto_area_de_operacao_vale: string (nullable = true)
 |-- contrato_vigente: string (nullable = true)
 |-- pacote: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cnpj_fornecedor: string (nullable = true)
 |-- conta_do_fornecedor_sap: string (nullable = true)
 |-- fornecedor: string (nullable = true)
 |-- objeto_contratual: string (nullable = true)
 |-- prazo_analise: string (nullable = true)
 |-- prazo_pagamento: string (nullable = true)
 |-- status_medicao: string (nullable = true)
 |-- n_do_bm: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- tecnico_de_medicao: string (nullable = true)
 |-- data_bm_preenchido: string (nullable = true)
 |-- data_analise_finalizada: string (nullable = true)
 |-- tempo_gasto_para_analise: string (nullable = true)
 |-- fiscal_nomeado: string (nullable = true)
 |-- data_aprovacao_fiscal: string (nullable = true)
 |-- tempo_gasto_p_aprovacao_do_fiscal: string (nullable = true)
 |-- administrador_vale:

In [12]:
from src.core.session import SparkFactory
from src.bronze.converters.pandas_to_spark import PandasToSparkConverter

spark = SparkFactory.create()

spark_df = PandasToSparkConverter.convert(
    spark=spark,
    pandas_df=df
)

spark_df.printSchema()

root
 |-- projeto_area_de_operacao_vale: string (nullable = true)
 |-- contrato_vigente: string (nullable = true)
 |-- pacote: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cnpj_fornecedor: string (nullable = true)
 |-- conta_do_fornecedor_sap: string (nullable = true)
 |-- fornecedor: string (nullable = true)
 |-- objeto_contratual: string (nullable = true)
 |-- prazo_analise: string (nullable = true)
 |-- prazo_pagamento: string (nullable = true)
 |-- status_medicao: string (nullable = true)
 |-- n_do_bm: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- tecnico_de_medicao: string (nullable = true)
 |-- data_bm_preenchido: string (nullable = true)
 |-- data_analise_finalizada: string (nullable = true)
 |-- tempo_gasto_para_analise: string (nullable = true)
 |-- fiscal_nomeado: string (nullable = true)
 |-- data_aprovacao_fiscal: string (nullable = true)
 |-- tempo_gasto_p_aprovacao_do_fiscal: string (nullable = true)
 |-- administrador_vale:

In [13]:
#Próximo passo: enriquecer spark_df com metadados Bronze:

from src.bronze.metadata import enrich_with_bronze_metadata

bronze_df = enrich_with_bronze_metadata(
    df=spark_df,
    snapshot_date="2024-07-13_0800",
    source_file="ControleMedicoesPagamentos.csv",
    source_path=source_path,
    source_type="csv",
    file_hash="manual_test_hash"
)

bronze_df.printSchema()
bronze_df.show(5)

root
 |-- projeto_area_de_operacao_vale: string (nullable = true)
 |-- contrato_vigente: string (nullable = true)
 |-- pacote: string (nullable = true)
 |-- contrato: string (nullable = true)
 |-- cnpj_fornecedor: string (nullable = true)
 |-- conta_do_fornecedor_sap: string (nullable = true)
 |-- fornecedor: string (nullable = true)
 |-- objeto_contratual: string (nullable = true)
 |-- prazo_analise: string (nullable = true)
 |-- prazo_pagamento: string (nullable = true)
 |-- status_medicao: string (nullable = true)
 |-- n_do_bm: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- tecnico_de_medicao: string (nullable = true)
 |-- data_bm_preenchido: string (nullable = true)
 |-- data_analise_finalizada: string (nullable = true)
 |-- tempo_gasto_para_analise: string (nullable = true)
 |-- fiscal_nomeado: string (nullable = true)
 |-- data_aprovacao_fiscal: string (nullable = true)
 |-- tempo_gasto_p_aprovacao_do_fiscal: string (nullable = true)
 |-- administrador_vale:

In [14]:
from src.bronze.writers.parquet_writer import BronzeParquetWriter

writer = BronzeParquetWriter(
    bucket_name="contracts",
    bronze_prefix="bronze"
)

output_path = writer.write(
    df=bronze_df,
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date="2024-07-13_0800",
    source_type="csv",
    mode="overwrite"
)

print(output_path)

26/06/22 23:37:05 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


s3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-07-13_0800/


In [15]:
from src.bronze.pipeline import BronzePipeline

pipeline = BronzePipeline()

output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv",
    source_type="csv",
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date="2024-07-13_0800",
    source_file="ControleMedicoesPagamentos.csv",
    file_hash="manual_test_hash"
)

print(output_path)

ImportError: cannot import name 'BronzePipeline' from 'src.bronze.pipeline' (/app/src/bronze/pipeline.py)

In [16]:
bronze_df = pipeline.spark.read.parquet(output_path)

bronze_df.printSchema()
bronze_df.show(5)

NameError: name 'pipeline' is not defined